In [ ]:
import os, re, glob, random, math, time, shutil, subprocess, tarfile, zipfile, warnings
import numpy as np, pandas as pd
import requests
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda"

# ---------------- configuration ----------------
SEED = 42
S = 256
BATCH = 16
EPOCHS = 50
LR, WD = 1e-3, 1e-2
SPLIT = (0.7, 0.1, 0.2)

LAMBDA_CLS, LAMBDA_CAM, LAMBDA_ACE = 0.2, 0.3, 0.5
M_PRIORS = 5
PRIOR_RES = 64
ACE_REGION = 4
N_BINS = 10


IMG_GLOB = None
MASK_GLOB = None
LABEL_FILE, LABEL_NAME_COL, LABEL_COL = None, None, None

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print("device:", device)

In [ ]:
ART_ID, FILE_ID = 32672274, 65547621
DATA = "/content/pu2756"; os.makedirs(DATA, exist_ok=True)

def extract(path, dst):
    n = path.lower()
    if n.endswith(".zip"): zipfile.ZipFile(path).extractall(dst)
    elif n.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2")): tarfile.open(path).extractall(dst)
    elif n.endswith(".7z"):
        subprocess.run(["pip", "-q", "install", "py7zr"], check=True); import py7zr; py7zr.SevenZipFile(path).extractall(dst)
    elif n.endswith(".rar"):
        subprocess.run("apt-get -qq install -y unrar", shell=True); subprocess.run(["unrar", "x", "-o+", path, dst + "/"])
    else: return False
    return True

if not glob.glob(DATA + "/**/*.*", recursive=True):
    info = requests.get(f"https://api.figshare.com/v2/articles/{ART_ID}", timeout=60).json()
    files = info["files"]
    for f in files: print(f["id"], f["name"], f"{f['size'] / 1e6:.0f} MB")
    for f in [f for f in files if f["id"] == FILE_ID] or files:
        dst = os.path.join(DATA, f["name"])
        with requests.get(f["download_url"], stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dst, "wb") as fh:
                for chunk in r.iter_content(1 << 20): fh.write(chunk)
        if extract(dst, DATA): os.remove(dst)

all_files = sorted(p for p in glob.glob(DATA + "/**/*", recursive=True) if os.path.isfile(p))
print(len(all_files), "files;", pd.Series([os.path.splitext(p)[1].lower() for p in all_files]).value_counts().to_dict())
print(*[os.path.relpath(p, DATA) for p in all_files[:6]], sep="\n")

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
MASK_TOKEN = re.compile(r"(mask|label|gt|annot|ground)", re.I)
norm_stem = lambda p: re.sub(r"([_\-\s]?(mask|label|seg|gt|annotation)s?)+$", "", os.path.splitext(os.path.basename(p))[0].lower())

if IMG_GLOB and MASK_GLOB:
    img_paths, msk_paths = sorted(glob.glob(IMG_GLOB, recursive=True)), sorted(glob.glob(MASK_GLOB, recursive=True))
else:
    pics = [p for p in all_files if p.lower().endswith(IMG_EXT)]
    is_mask = lambda p: bool(MASK_TOKEN.search(os.path.relpath(os.path.dirname(p), DATA))) or bool(MASK_TOKEN.search(os.path.splitext(os.path.basename(p))[0]))
    msk_paths = [p for p in pics if is_mask(p)]; img_paths = [p for p in pics if not is_mask(p)]
msk_by = {}
for p in msk_paths: msk_by.setdefault(norm_stem(p), p)
pairs = [(p, msk_by[norm_stem(p)]) for p in img_paths if norm_stem(p) in msk_by]
print(f"images: {len(img_paths)} | masks: {len(msk_paths)} | matched pairs: {len(pairs)}")
assert pairs, "No image/mask pairs found - set IMG_GLOB / MASK_GLOB in the configuration cell."
print("example pair:", os.path.relpath(pairs[0][0], DATA), "<->", os.path.relpath(pairs[0][1], DATA))

# ---- benign / malignant label ----
def labels_from_paths(pairs):
    out = []
    for ip, _ in pairs:
        r = os.path.relpath(ip, DATA).lower()
        out.append(1 if "malignant" in r else 0 if "benign" in r else None)
    return out if all(v is not None for v in out) else None

def labels_from_table(pairs):
    stems = {norm_stem(ip): k for k, (ip, _) in enumerate(pairs)}
    tables = [LABEL_FILE] if LABEL_FILE else [p for p in all_files if p.lower().endswith((".csv", ".xlsx", ".xls"))]
    for tp in tables:
        try: df = pd.read_csv(tp) if tp.lower().endswith(".csv") else pd.read_excel(tp)
        except Exception: continue
        name_cols = [LABEL_NAME_COL] if LABEL_NAME_COL else list(df.columns)
        for nc in name_cols:
            keys = df[nc].astype(str).map(lambda s: norm_stem(s))
            if keys.isin(set(stems)).mean() < 0.5: continue
            cand = [LABEL_COL] if LABEL_COL else [c for c in df.columns if c != nc and re.search(r"malig|benign|patholog|diagnos|label|class|type|nature", str(c), re.I)]
            for lc in cand:
                col = df[lc]
                if col.nunique() != 2: continue
                numeric = pd.api.types.is_numeric_dtype(col)
                pos = (col == col.max()) if numeric else col.astype(str).str.contains("malig", case=False)
                lab = dict(zip(keys, pos.astype(int)))
                if all(s in lab for s in stems):
                    print(f"labels from {os.path.basename(tp)} | name col '{nc}' | label col '{lc}' | values {col.value_counts().to_dict()} | positive (malignant) = "
                          f"{'larger value (' + str(col.max()) + ')' if numeric else 'strings containing malig'}")
                    return [lab[norm_stem(ip)] for ip, _ in pairs]
    return None

lab = labels_from_paths(pairs); src = "folder names"
if lab is None: lab, src = labels_from_table(pairs), "table"
if lab is None:
    print("WARNING: no benign/malignant label found -> using 'mask non-empty' as image-level target (constant here, CH uninformative). Set LABEL_FILE/LABEL_COL.")
    lab_mode = "mask"
else:
    lab_mode = "label"; print(f"image-level target = malignant (from {src}); counts:", pd.Series(lab).value_counts().to_dict())

In [ ]:
def to_square(a):
    h, w = a.shape; m = max(h, w); out = np.zeros((m, m), a.dtype)
    y0, x0 = (m - h) // 2, (m - w) // 2; out[y0:y0 + h, x0:x0 + w] = a; return out

def load_pair(k):
    ip, mp = pairs[k]
    im = np.asarray(Image.open(ip).convert("L"))
    m = np.asarray(Image.open(mp).convert("L")); m = (m > 127) if m.max() > 1 else (m > 0)
    if m.shape != im.shape: m = np.asarray(Image.fromarray(m.astype(np.uint8) * 255).resize(im.shape[::-1], Image.NEAREST)) > 127
    img = np.asarray(Image.fromarray(to_square(im)).resize((S, S), Image.BILINEAR))
    msk = np.asarray(Image.fromarray(to_square(m.astype(np.uint8))).resize((S, S), Image.NEAREST))
    return img, msk

with ThreadPoolExecutor(8) as ex: res = list(tqdm(ex.map(load_pair, range(len(pairs))), total=len(pairs)))
IMGS = torch.from_numpy(np.stack([r[0] for r in res])); MSKS = torch.from_numpy(np.stack([r[1] for r in res]))
CLS = torch.tensor(lab if lab_mode == "label" else [int(m.any()) for m in MSKS.numpy()]).float()
print(IMGS.shape, "| lesion area fraction (mean): %.3f" % MSKS.float().mean(), "| empty masks:", int((MSKS.flatten(1).sum(1) == 0).sum()))

# stratified split
rng = np.random.RandomState(SEED); split_idx = {"train": [], "val": [], "test": []}
for cval in CLS.unique():
    ids = np.where(CLS.numpy() == cval.item())[0]; rng.shuffle(ids); n = len(ids)
    a, b = int(SPLIT[0] * n), int((SPLIT[0] + SPLIT[1]) * n)
    split_idx["train"] += ids[:a].tolist(); split_idx["val"] += ids[a:b].tolist(); split_idx["test"] += ids[b:].tolist()
arrs = {k: (IMGS[v], MSKS[v], CLS[v]) for k, v in split_idx.items()}
for k, a in arrs.items(): print(k, len(a[0]), "positives:", int(a[2].sum()))

def random_affine(x, y):                     # x,y: (1,S,S); small rotation / scale / shift, same transform for image and mask
    ang, sc = math.radians(random.uniform(-10, 10)), random.uniform(0.9, 1.1)
    tx, ty = random.uniform(-0.08, 0.08), random.uniform(-0.08, 0.08)
    c_, s_ = math.cos(ang) / sc, math.sin(ang) / sc
    grid = F.affine_grid(torch.tensor([[[c_, -s_, tx], [s_, c_, ty]]], dtype=torch.float32), (1, 1, S, S), align_corners=False)
    return (F.grid_sample(x[None], grid, mode="bilinear", align_corners=False)[0],
            F.grid_sample(y[None], grid, mode="nearest", align_corners=False)[0])

class SegDS(Dataset):
    def __init__(self, a, train=False): self.imgs, self.masks, self.cls = a; self.train = train
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = self.imgs[i].float().div(255.)[None]; y = self.masks[i].float()[None]
        if self.train:
            if random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
            if random.random() < 0.5: x, y = random_affine(x, y)
            x = x * (1 + 0.2 * (random.random() - 0.5)) + 0.1 * (random.random() - 0.5)
        x = (x - x.mean()) / (x.std() + 1e-6)
        return x, y, self.cls[i], i

train_ds, val_ds, test_ds = SegDS(arrs["train"], True), SegDS(arrs["val"]), SegDS(arrs["test"])
mk = lambda ds, sh: DataLoader(ds, BATCH, shuffle=sh, num_workers=2, drop_last=sh, pin_memory=True)
train_loader, val_loader, test_loader = mk(train_ds, True), mk(val_ds, False), mk(test_ds, False)

In [ ]:
class BNAct(nn.Sequential):
    def __init__(self, i, o, k=1, s=1, groups=1, act=True):
        l = [nn.Conv2d(i, o, k, s, k // 2, groups=groups, bias=False), nn.BatchNorm2d(o)]
        super().__init__(*l, nn.SiLU(inplace=True)) if act else super().__init__(*l)

class IRBlock(nn.Module):                         # MBConv-style inverted residual
    def __init__(self, cin, cout, stride=1, e=4):
        super().__init__(); mid = cin * e
        self.body = nn.Sequential(BNAct(cin, mid, 1), BNAct(mid, mid, 3, stride, groups=mid), BNAct(mid, cout, 1, act=False))
        self.res = stride == 1 and cin == cout
    def forward(self, x):
        y = self.body(x); return x + y if self.res else y

def q_shift(x, H, W):                             # x: (B,T,C); shift quarter-channel groups by one pixel in 4 directions
    B, T, C = x.shape; z = x.transpose(1, 2).reshape(B, C, H, W); c = C // 4; o = torch.zeros_like(z)
    o[:, :c, :, 1:] = z[:, :c, :, :-1]; o[:, c:2 * c, :, :-1] = z[:, c:2 * c, :, 1:]
    o[:, 2 * c:3 * c, 1:, :] = z[:, 2 * c:3 * c, :-1, :]; o[:, 3 * c:4 * c, :-1, :] = z[:, 3 * c:4 * c, 1:, :]
    o[:, 4 * c:] = z[:, 4 * c:]
    return o.reshape(B, C, T).transpose(1, 2)

class BiWKV(nn.Module):
    def __init__(self, C):
        super().__init__()
        w0 = torch.exp(torch.linspace(math.log(0.2), math.log(12.0), C))          # decay speeds, softplus-parametrised
        self.w = nn.Parameter(torch.log(torch.expm1(w0))); self.u = nn.Parameter(torch.zeros(C))
    def forward(self, k, v):                                                      # (B,T,C) -> (B,T,C)
        B, T, C = k.shape
        with torch.autocast("cuda", enabled=False):
            k, v = k.float(), v.float(); w = F.softplus(self.w.float())
            idx = torch.arange(T, device=k.device); dist = (idx[:, None] - idx[None, :]).abs().float()
            A = torch.exp(-(dist.clamp(min=1) - 1)[None] / T * w[:, None, None])  # (C,T,T)
            eye = torch.eye(T, dtype=torch.bool, device=k.device)[None]
            A = torch.where(eye, torch.exp(self.u.float())[:, None, None].expand(C, T, T), A)
            ek = torch.exp((k - k.amax(1, keepdim=True)).clamp(min=-60))          # shift cancels in the ratio (stability)
            num = torch.einsum("cti,bic->btc", A, ek * v); den = torch.einsum("cti,bic->btc", A, ek)
            return num / (den + 1e-8)

class SpatialMix(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.mu_k, self.mu_v, self.mu_r = (nn.Parameter(torch.full((1, 1, C), 0.5)) for _ in range(3))
        self.key, self.value, self.recept, self.out = (nn.Linear(C, C, bias=False) for _ in range(4))
        self.ln, self.wkv = nn.LayerNorm(C), BiWKV(C)
    def forward(self, x, H, W):
        xx = q_shift(x, H, W)
        k = self.key(x * self.mu_k + xx * (1 - self.mu_k)); v = self.value(x * self.mu_v + xx * (1 - self.mu_v))
        r = self.recept(x * self.mu_r + xx * (1 - self.mu_r))
        return self.out(torch.sigmoid(r) * self.ln(self.wkv(k, v)))

class ChannelMix(nn.Module):
    def __init__(self, C, ratio=2):
        super().__init__()
        self.mu_k, self.mu_r = (nn.Parameter(torch.full((1, 1, C), 0.5)) for _ in range(2))
        self.key, self.value, self.recept = nn.Linear(C, C * ratio, bias=False), nn.Linear(C * ratio, C, bias=False), nn.Linear(C, C, bias=False)
    def forward(self, x, H, W):
        xx = q_shift(x, H, W)
        k = torch.relu(self.key(x * self.mu_k + xx * (1 - self.mu_k))) ** 2
        return torch.sigmoid(self.recept(x * self.mu_r + xx * (1 - self.mu_r))) * self.value(k)

class VRWKVBlock(nn.Module):
    def __init__(self, C):
        super().__init__(); self.ln1, self.ln2 = nn.LayerNorm(C), nn.LayerNorm(C); self.att, self.ffn = SpatialMix(C), ChannelMix(C)
    def forward(self, x, H, W):
        x = x + self.att(self.ln1(x), H, W); return x + self.ffn(self.ln2(x), H, W)

class IRRWKV(nn.Module):                          # expand -> DWConv -> VRWKV -> project (+ residual)
    def __init__(self, C, e=2):
        super().__init__(); mid = C * e
        self.expand, self.dw = BNAct(C, mid, 1), BNAct(mid, mid, 3, 1, groups=mid)
        self.rwkv = VRWKVBlock(mid); self.proj = nn.Sequential(nn.Conv2d(mid, C, 1, bias=False), nn.BatchNorm2d(C))
    def forward(self, x):
        y = self.dw(self.expand(x)); B, Cm, H, W = y.shape
        y = self.rwkv(y.flatten(2).transpose(1, 2), H, W).transpose(1, 2).reshape(B, Cm, H, W)
        return x + self.proj(y)

def resize_to(f, size):
    if f.shape[-2:] == size: return f
    return F.adaptive_avg_pool2d(f, size) if f.shape[-1] > size[-1] else F.interpolate(f, size=size, mode="bilinear", align_corners=False)

class CCM(nn.Module):                             # cross-channel mix over neighbouring-scale skip features
    def __init__(self, in_chs, out_ch):
        super().__init__()
        self.mix, self.dw = BNAct(sum(in_chs), out_ch, 1), BNAct(out_ch, out_ch, 3, groups=out_ch)
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(out_ch, max(out_ch // 4, 8), 1), nn.SiLU(), nn.Conv2d(max(out_ch // 4, 8), out_ch, 1), nn.Sigmoid())
    def forward(self, feats, size):
        y = self.dw(self.mix(torch.cat([resize_to(f, size) for f in feats], 1))); return y * self.se(y)

class RWKVUNet2D(nn.Module):
    def __init__(self, use_cls=False, use_cam=False, feats=(16, 32, 48, 96, 160, 256)):
        super().__init__()
        assert use_cls or not use_cam, "CAM needs the classification head"
        self.use_cls, self.use_cam = use_cls, use_cam
        f = feats; n = len(f)
        self.enc = nn.ModuleList([
            nn.Sequential(BNAct(1, f[0], 3), BNAct(f[0], f[0], 3)),                                   # 1/1
            nn.Sequential(IRBlock(f[0], f[1], 2, 2)),                                                 # 1/2
            nn.Sequential(IRBlock(f[1], f[2], 2), IRBlock(f[2], f[2])),                               # 1/4
            nn.Sequential(IRBlock(f[2], f[3], 2), IRBlock(f[3], f[3]), IRBlock(f[3], f[3])),          # 1/8
            nn.Sequential(IRBlock(f[3], f[4], 2), IRRWKV(f[4]), IRRWKV(f[4])),                        # 1/16  (IR-RWKV)
            nn.Sequential(IRBlock(f[4], f[5], 2), IRRWKV(f[5]), IRRWKV(f[5]))])                       # 1/32  (IR-RWKV)
        self.nb = {l: [j for j in (l - 1, l, l + 1) if 0 <= j < n] for l in range(n - 1)}             # neighbour levels for CCM
        self.ccm = nn.ModuleList([CCM([f[j] for j in self.nb[l]], f[l]) for l in range(n - 1)])
        self.ups = nn.ModuleList([nn.ConvTranspose2d(f[l + 1], f[l], 2, 2) for l in range(n - 2, -1, -1)])
        self.decs = nn.ModuleList([nn.Sequential(BNAct(2 * f[l], f[l], 3), BNAct(f[l], f[l], 3)) for l in range(n - 2, -1, -1)])
        if use_cls: self.fc = nn.Linear(f[-1], 1)
        if use_cam: self.refine = BNAct(f[0] + 1, f[0], 3)
        self.head = nn.Conv2d(f[0], 1, 1)

    def forward(self, x):
        size = x.shape[-2:]; skips = []
        for e in self.enc:
            x = e(x); skips.append(x)
        bott = skips[-1]; d = bott; n = len(skips)
        for j, l in enumerate(range(n - 2, -1, -1)):
            s = self.ccm[l]([skips[k] for k in self.nb[l]], skips[l].shape[-2:])
            d = self.decs[j](torch.cat([self.ups[j](d), s], 1))
        out = {}
        if self.use_cls:
            logit = self.fc(bott.mean((2, 3)))
            cam = F.relu(torch.einsum("bchw,c->bhw", bott, self.fc.weight[0])).unsqueeze(1)
            cam = cam / (cam.amax((2, 3), keepdim=True) + 1e-6)
            cam = F.interpolate(cam, size=size, mode="bilinear", align_corners=False)
            out["cls"], out["cam"] = logit, cam
            if self.use_cam: d = self.refine(torch.cat([d, cam * torch.sigmoid(logit).view(-1, 1, 1, 1)], 1))
        out["logits"] = self.head(d)
        return out

_m = RWKVUNet2D(True, True).to(device)
with torch.no_grad(): _o = _m(torch.randn(2, 1, S, S, device=device))
print({k: tuple(v.shape) for k, v in _o.items()}, "| params: %.1fM" % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o

In [ ]:
def dice_loss(logits, y, smooth=1.0):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + y.sum((1, 2, 3))
    return (1 - (2 * inter + smooth) / (den + smooth)).mean()

def seg_loss(logits, y):                       # L_seg = 1/2 Dice + 1/2 BCE
    return 0.5 * dice_loss(logits, y) + 0.5 * F.binary_cross_entropy_with_logits(logits, y)

def cam_loss(cam, y, c):                       # align CAM with the lesion mask on images with a positive image-level target
    pos = c > 0.5
    if pos.sum() == 0: return cam.sum() * 0
    return F.binary_cross_entropy(cam[pos].clamp(1e-4, 1 - 1e-4), y[pos])

class PriorBank:
    def __init__(self, masks, res=PRIOR_RES):
        self.res = res
        self.bank = F.adaptive_avg_pool2d(masks[:, None].float(), res).flatten(1).to(device)
        self.norm = F.normalize(self.bank, dim=1)
    @torch.no_grad()
    def retrieve(self, prob, M, exclude=None):
        q = F.adaptive_avg_pool2d(prob, self.res).flatten(1)
        sim = F.normalize(q, dim=1) @ self.norm.T
        if exclude is not None: sim[torch.arange(len(q), device=sim.device), exclude.to(sim.device)] = -1
        w, idx = sim.topk(M, dim=1)
        return w.clamp(0, 1), self.bank[idx]

def uncertainty(prob, bank, M=M_PRIORS, exclude=None):
    prob = prob.float()
    w, priors = bank.retrieve(prob.detach(), M, exclude)
    q = F.adaptive_avg_pool2d(prob, bank.res).flatten(1)
    U = (w[:, :, None] * (priors - q[:, None]).abs()).sum(1) / M
    return U.view(-1, 1, bank.res, bank.res)

def ace_loss(U, prob, y, res=PRIOR_RES, region=ACE_REGION):
    e = (F.adaptive_avg_pool2d(prob, res) - F.adaptive_avg_pool2d(y, res)).abs().detach()
    return (F.avg_pool2d(U, region) - F.avg_pool2d(e, region)).abs().mean()

bank = PriorBank(arrs["train"][1])
print("prior bank:", tuple(bank.bank.shape))

In [ ]:
def autocast(): return torch.autocast("cuda", enabled=AMP)

@torch.no_grad()
def val_dice(model, loader):
    model.eval(); ds = []
    for x, y, c, i in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        pb = (torch.sigmoid(out["logits"].float()) > 0.5).float()
        inter = (pb * y).sum((1, 2, 3))
        ds.append((2 * inter + 1) / (pb.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1))
    return torch.cat(ds).mean().item()

@torch.no_grad()
def evaluate(model, loader, bank, n_vis=0):
    model.eval()
    per = {k: [] for k in ("dice", "iou", "prec", "rec")}
    TP = FP = FN = 0.0; nb = N_BINS
    cnt = torch.zeros(2, nb, device=device, dtype=torch.float64); sconf = torch.zeros_like(cnt); sacc = torch.zeros_like(cnt)
    g = torch.zeros(6, dtype=torch.float64, device=device)      # n, sum u, sum e, sum u^2, sum e^2, sum ue
    uerr = uok = nerr = nok = 0.0
    pcorr, img_u, img_e, vis, cp, ct = [], [], [], [], [], []
    for x, y, c, idx in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        prob = torch.sigmoid(out["logits"].float()); pb = (prob > 0.5).float()
        inter = (pb * y).sum((1, 2, 3)); ps = pb.sum((1, 2, 3)); ys = y.sum((1, 2, 3))
        dice = (2 * inter + 1) / (ps + ys + 1)
        per["dice"].append(dice); per["iou"].append((inter + 1) / (ps + ys - inter + 1))
        per["prec"].append((inter + 1) / (ps + 1)); per["rec"].append((inter + 1) / (ys + 1))
        TP += inter.sum().item(); FP += (ps - inter).sum().item(); FN += (ys - inter).sum().item()
        if "cls" in out: cp += torch.sigmoid(out["cls"].float()).squeeze(1).tolist(); ct += c.tolist()
        for k, (conf, ind) in enumerate([(prob, y), (1 - prob, 1 - y)]):          # calibration bins: fg, bg
            b = (conf * nb).long().clamp(0, nb - 1).flatten()
            cnt[k] += torch.bincount(b, minlength=nb).double()
            sconf[k] += torch.bincount(b, weights=conf.flatten().double(), minlength=nb)
            sacc[k] += torch.bincount(b, weights=ind.flatten().double(), minlength=nb)
        U = uncertainty(prob, bank)                                               # uncertainty vs. error
        err = F.adaptive_avg_pool2d((pb != y).float(), bank.res)
        u, e = U.flatten(1).double(), err.flatten(1).double()
        uc, ec = u - u.mean(1, keepdim=True), e - e.mean(1, keepdim=True)
        den = uc.norm(dim=1) * ec.norm(dim=1); ok = den > 1e-8
        pcorr += ((uc * ec).sum(1)[ok] / den[ok]).tolist()
        has = e > 0
        uerr += u[has].sum().item(); nerr += has.sum().item(); uok += u[~has].sum().item(); nok += (~has).sum().item()
        g += torch.stack([torch.tensor(float(u.numel()), device=device, dtype=torch.float64), u.sum(), e.sum(), (u * u).sum(), (e * e).sum(), (u * e).sum()])
        img_u += u.mean(1).tolist(); img_e += (1 - dice).tolist()
        if len(vis) < n_vis:
            cam = out.get("cam", None)
            for j in range(len(x)):
                if len(vis) >= n_vis: break
                vis.append(dict(x=x[j, 0].cpu(), y=y[j, 0].cpu(), p=prob[j, 0].cpu(), c=int(c[j]),
                                cam=None if cam is None else cam[j, 0].float().cpu(),
                                U=F.interpolate(U[j:j + 1], size=x.shape[-2:], mode="bilinear")[0, 0].cpu()))
    n, su, se, suu, see, sue = g.tolist()
    cov = sue / n - (su / n) * (se / n)
    gcorr = cov / (math.sqrt(max(suu / n - (su / n) ** 2, 1e-12)) * math.sqrt(max(see / n - (se / n) ** 2, 1e-12)))
    valid = cnt > 0
    o, ex = sacc / cnt.clamp(min=1), sconf / cnt.clamp(min=1)
    ace = (((o - ex).abs() * valid).sum() / valid.sum()).item()
    m = {k: torch.cat(v).mean().item() for k, v in per.items()}
    res = {"Dice": m["dice"], "IoU": m["iou"], "Precision": m["prec"], "Recall": m["rec"],
           "F1": 2 * TP / (2 * TP + FP + FN + 1e-9),                              # pooled over all pixels
           "ACE": ace, "U_mean": su / n, "U_on_error": uerr / max(nerr, 1), "U_on_correct": uok / max(nok, 1),
           "corr_U_err_pixel": gcorr, "corr_U_err_per_image": float(np.mean(pcorr)) if pcorr else float("nan"),
           "corr_meanU_vs_(1-Dice)": float(np.corrcoef(img_u, img_e)[0, 1])}
    if cp:
        res["cls_acc"] = float(np.mean((np.array(cp) > 0.5) == (np.array(ct) > 0.5)))
        res["cls_auc"] = float(roc_auc_score(ct, cp)) if len(set(ct)) > 1 else float("nan")
    return res, vis

In [ ]:
CKPT_DIR = "/content/ckpt_pu"; os.makedirs(CKPT_DIR, exist_ok=True)

VARIANTS = {   # name: (use_CH, use_CAM, use_ACE)
    "RWKV-UNet":             (False, False, False),
    "RWKV-UNet+CH":          (True,  False, False),
    "RWKV-UNet+CH+CAM":      (True,  True,  False),
    "RWKV-UNet+CH+CAM+ACE":  (True,  True,  True),
}

def train_variant(name, use_ch, use_cam, use_ace):
    set_seed(SEED)
    model = RWKVUNet2D(use_ch, use_cam).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, path, hist = -1, f"{CKPT_DIR}/{name}.pt", []
    for ep in range(EPOCHS):
        for gp in opt.param_groups: gp["lr"] = LR * (1 - ep / EPOCHS) ** 0.9          # poly schedule
        model.train(); agg = {"tot": 0, "seg": 0, "cls": 0, "cam": 0, "ace": 0}; nit = 0
        for x, y, c, idx in tqdm(train_loader, desc=f"{name} ep{ep + 1}/{EPOCHS}", leave=False):
            x, y, c = x.to(device), y.to(device), c.to(device)
            with autocast(): out = model(x)
            logits = out["logits"].float(); prob = torch.sigmoid(logits)
            Ls = seg_loss(logits, y); L = Ls; Lc = Lm = La = torch.zeros((), device=device)
            if use_ch:
                Lc = F.binary_cross_entropy_with_logits(out["cls"].float().squeeze(1), c); L = L + LAMBDA_CLS * Lc
            if use_cam:
                Lm = cam_loss(out["cam"].float(), y, c); L = L + LAMBDA_CAM * Lm
            if use_ace:
                U = uncertainty(prob, bank, M_PRIORS, exclude=idx)
                La = ace_loss(U, prob, y); L = L + LAMBDA_ACE * La
            opt.zero_grad(set_to_none=True)
            scaler.scale(L).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            scaler.step(opt); scaler.update()
            for k, v in zip(agg, (L, Ls, Lc, Lm, La)): agg[k] += float(v)
            nit += 1
        vd = val_dice(model, val_loader)
        hist.append({"epoch": ep + 1, "val_dice": vd, **{k: v / nit for k, v in agg.items()}})
        print(f"[{name}] ep{ep + 1:02d} loss {agg['tot'] / nit:.4f} (seg {agg['seg'] / nit:.3f} cls {agg['cls'] / nit:.3f} "
              f"cam {agg['cam'] / nit:.3f} ace {agg['ace'] / nit:.3f}) | val Dice {vd:.4f}")
        if vd > best: best = vd; torch.save(model.state_dict(), path)
    model.load_state_dict(torch.load(path))               # best-on-VALIDATION checkpoint
    return model, pd.DataFrame(hist)

models, histories, results, vis_store = {}, {}, {}, {}
for name, flags in VARIANTS.items():
    t0 = time.time()
    models[name], histories[name] = train_variant(name, *flags)
    results[name], vis_store[name] = evaluate(models[name], test_loader, bank, n_vis=8)
    print(f"== {name}: done in {(time.time() - t0) / 60:.1f} min | test Dice {results[name]['Dice']:.4f}\n")

In [ ]:
res_df = pd.DataFrame(results).T
seg_cols = ["Dice", "IoU", "Precision", "Recall", "F1"]
unc_cols = ["ACE", "U_mean", "U_on_error", "U_on_correct", "corr_U_err_pixel", "corr_U_err_per_image", "corr_meanU_vs_(1-Dice)"]
print("Segmentation (test)"); display(res_df[seg_cols].round(4))
print("Calibration / uncertainty (test)"); display(res_df[unc_cols].round(4))
if "cls_auc" in res_df: print("Image-level classification head (test)"); display(res_df[["cls_acc", "cls_auc"]].dropna().round(4))
res_df.to_csv("/content/pu2756_ablation_results.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for n_, h in histories.items():
    ax[0].plot(h.epoch, h.seg, label=n_); ax[1].plot(h.epoch, h.val_dice, label=n_)
ax[0].set_title("train L_seg"); ax[1].set_title("val Dice"); ax[1].legend(fontsize=7)
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
def show_qualitative(name, n=6):
    v = vis_store[name][:n]
    cols = ["Image", "Ground truth", "Prediction", "CAM", "Uncertainty U", "Error |pred-GT|"]
    fig, axs = plt.subplots(len(v), 6, figsize=(14, 2.3 * len(v))); axs = np.atleast_2d(axs)
    for r, s in enumerate(v):
        axs[r, 0].imshow(s["x"], cmap="gray")
        axs[r, 1].imshow(s["x"], cmap="gray"); axs[r, 1].imshow(s["y"], cmap="Greens", alpha=0.45)
        axs[r, 2].imshow(s["x"], cmap="gray"); axs[r, 2].imshow((s["p"] > 0.5).float(), cmap="Reds", alpha=0.45)
        axs[r, 3].imshow(s["x"], cmap="gray")
        if s["cam"] is not None: axs[r, 3].imshow(s["cam"], cmap="jet", alpha=0.5, vmin=0, vmax=1)
        im = axs[r, 4].imshow(s["U"], cmap="magma"); plt.colorbar(im, ax=axs[r, 4], fraction=0.046)
        axs[r, 5].imshow((s["p"] > 0.5).float().sub(s["y"]).abs(), cmap="gray", vmin=0, vmax=1)
        for a in axs[r]: a.axis("off")
    for j, t in enumerate(cols): axs[0, j].set_title(t, fontsize=9)
    fig.suptitle(name); plt.tight_layout(); plt.show()

show_qualitative("RWKV-UNet+CH+CAM+ACE")
